In [2]:
from datasets import load_dataset

ds_2024 = load_dataset("Reubencf/2024_events")
ds_2025 = load_dataset("Reubencf/2025_events")

from datasets import concatenate_datasets

combined_events = concatenate_datasets([
    ds_2024["train"],
    ds_2025["train"]
])

# convert section to lowercase because they are inconsistent across both datasets
events = combined_events.map(
    lambda x: {"section": x["section"].lower()}
)


In [3]:
print("\nColumns:")
print(events.column_names)
print("\nFeatures:")
print(events.features)
print(f"\nTotal rows: {len(events)}")

import random
from pprint import pprint

n_samples = 1

indices = random.sample(range(len(events)), n_samples)

print(f"\n{n_samples} Random events:")
for idx in indices:
    sample = events[idx]
    print(f"\nSample[{idx}]:")
    pprint(sample)

# count articles in each section
from collections import Counter

section_counts = Counter(events["section"])

print("\nSection counts:")
for section, count in section_counts.most_common():
    print(f"\t{section}: {count}")



Columns:
['month', 'year', 'day', 'section', 'content', 'instruction', 'response']

Features:
{'month': Value('string'), 'year': Value('int64'), 'day': Value('int64'), 'section': Value('string'), 'content': Value('string'), 'instruction': Value('string'), 'response': Value('string')}

Total rows: 10712

1 Random events:

Sample[8497]:
{'content': 'Thirteen people are killed and 50 others become ill, including 21 '
            'cases of blindness or impaired vision and some critically, from '
            'methanol poisoning in Kuwait. (Reuters)',
 'day': 13,
 'instruction': 'What caused the methanol poisoning outbreak in Kuwait in '
                'August 2025 that tragically killed 13 people and resulted in '
                '21 cases of blindness or impaired vision?',
 'month': 'August',
 'response': 'The tragic methanol poisoning outbreak that struck Kuwait in '
             'August 2025 was directly caused by the widespread distribution '
             'and consumption of **locally

In [4]:
import requests, time

ollama_base_url = "http://localhost:11434/api/"
embedding_models = [
    "mxbai-embed-large",
    "nomic-embed-text"
]

for embedding_model in embedding_models:

    start = time.time()
    print(f"\n**** Embeddings using model {embedding_model}")
    response = requests.post(
        ollama_base_url + "embeddings",
        json={
            "model": embedding_model,
            "prompt": sample["instruction"]
        }
    )
    end = time.time()
    print(f"{len(response.json()["embedding"])} dimensions generated in {round(end-start, 3)}s")


**** Embeddings using model mxbai-embed-large
1024 dimensions generated in 1.007s

**** Embeddings using model nomic-embed-text
768 dimensions generated in 0.621s


In [5]:
llm_models = [
    "qwen3.5:9b",
    "llama3.2:3b",
    "llama3.2:1b",
    "gpt-oss:20b"
]


for llm_model in llm_models:
    start = time.time()

    print(f"\n\n*** Trial generate with {llm_model}:")
    response = requests.post(
        ollama_base_url + "generate",
        json={
            "model": llm_model,
            "prompt": f"Summarize the following text clearly and concisely:\n\n{sample["content"]}",
            "stream": False
        }
    )
    print("Summary: " + response.json()["response"])
    end = time.time()
    print(f"Summary Time: {round(end - start)}s") 

    start = time.time()
    response = requests.post(
        ollama_base_url + "generate",
        json={
            "model": llm_model,
            "prompt": f"Give three main concise and clear points from the text:\n\n{sample["content"]}",
            "stream": False
        }
    )
    print("Main points:\n" + response.json()["response"])

    end = time.time()
    print(f"Main points time: {round(end - start)}s") 



*** Trial generate with qwen3.5:9b:
Summary: Methanol poisoning in Kuwait killed 13 people and left 50 others ill, including 21 cases of blindness or impaired vision.
Summary Time: 22s
Main points:
1. Thirteen people were killed from methanol poisoning in Kuwait.
2. Fifty others became ill.
3. Twenty-one of the victims suffered blindness or impaired vision.
Main points time: 45s


*** Trial generate with llama3.2:3b:
Summary: 13 people died and 50 were injured, with 21 experiencing blindness or impaired vision due to methanol poisoning in Kuwait.
Summary Time: 2s
Main points:
Here are three concise and clear points from the text:

1. Thirteen people were killed in an incident.
2. Fifty other individuals fell ill due to the incident, with severe consequences including blindness and impaired vision.
3. The cause of the illness was methanol poisoning.
Main points time: 1s


*** Trial generate with llama3.2:1b:
Summary: Thirteen people have died and 50 are injured, with many experiencing